# House Prices: EDA and Data Understanding

**Objective:** Understand the Kaggle House Prices data before preprocessing or modeling.

**Success criteria:**

- audit all 79 explanatory features;
- distinguish semantic absence from unknown missing values;
- analyze `SalePrice` on original and log scales;
- identify important relationships, redundancy, outliers, and train/test drift;
- export reusable figures and audit tables.

This notebook does not impute values, remove outliers, fit preprocessing, or train a model.


## 1. Setup and reproducibility

The notebook locates `Lab02` from either the repository root or the notebook directory. All generated artifacts are written under `outputs/`.


In [1]:
from pathlib import Path
import platform

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

SEED = 42
np.random.seed(SEED)

candidates = [Path.cwd(), Path.cwd() / "Week02" / "Lab02", Path.cwd().parent]
LAB_ROOT = next((p.resolve() for p in candidates if (p / "data" / "train.csv").exists()), None)
if LAB_ROOT is None:
    raise FileNotFoundError("Could not locate Week02/Lab02/data/train.csv")

DATA_DIR = LAB_ROOT / "data"
FIGURES_DIR = LAB_ROOT / "outputs" / "figures"
METRICS_DIR = LAB_ROOT / "outputs" / "metrics"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 160, "axes.titlesize": 12})

def save_figure(name: str) -> None:
    path = FIGURES_DIR / name
    plt.tight_layout()
    plt.savefig(path, bbox_inches="tight")
    print(f"Saved: {path.relative_to(LAB_ROOT)}")

print(f"Python: {platform.python_version()}")
print(f"pandas: {pd.__version__}; numpy: {np.__version__}; matplotlib: {matplotlib.__version__}")
print(f"Lab root: {LAB_ROOT}")


Python: 3.13.5
pandas: 3.0.5; numpy: 2.5.3; matplotlib: 3.11.2
Lab root: D:\Workspace\MonHoc\DL-Sgu\Week02\Lab02


## 2. Safe loading and integrity checks

The CSV files contain literal `NA` values that can mean either "feature absent" or genuinely unknown. They also contain the valid category `None` in `MasVnrType`. Reading a raw string-preserving copy prevents pandas from merging those meanings.


In [2]:
train_path = DATA_DIR / "train.csv"
test_path = DATA_DIR / "test.csv"

raw_train = pd.read_csv(train_path, keep_default_na=False)
raw_test = pd.read_csv(test_path, keep_default_na=False)
inferred_train = pd.read_csv(train_path)
inferred_test = pd.read_csv(test_path)

BASEMENT_COLUMNS = {"BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2"}
GARAGE_COLUMNS = {"GarageType", "GarageFinish", "GarageQual", "GarageCond"}
UNCONDITIONAL_ABSENCE_COLUMNS = {"Alley", "Fence", "MiscFeature"}
STRUCTURAL_ABSENCE = (
    BASEMENT_COLUMNS | GARAGE_COLUMNS | UNCONDITIONAL_ABSENCE_COLUMNS
    | {"FireplaceQu", "PoolQC"}
)
NUMERIC_STRUCTURAL_ABSENCE = {"GarageYrBlt"}

def build_structural_absence_masks(raw: pd.DataFrame) -> pd.DataFrame:
    numeric = lambda column: pd.to_numeric(raw[column], errors="coerce")
    basement_absent = numeric("TotalBsmtSF").eq(0)
    garage_absent = numeric("GarageCars").eq(0) & numeric("GarageArea").eq(0)
    fireplace_absent = numeric("Fireplaces").eq(0)
    pool_absent = numeric("PoolArea").eq(0)

    masks = {}
    for column in STRUCTURAL_ABSENCE | NUMERIC_STRUCTURAL_ABSENCE:
        raw_na = raw[column].eq("NA")
        if column in BASEMENT_COLUMNS:
            masks[column] = raw_na & basement_absent
        elif column in GARAGE_COLUMNS or column == "GarageYrBlt":
            masks[column] = raw_na & garage_absent
        elif column == "FireplaceQu":
            masks[column] = raw_na & fireplace_absent
        elif column == "PoolQC":
            masks[column] = raw_na & pool_absent
        else:
            masks[column] = raw_na
    return pd.DataFrame(masks, index=raw.index).astype(bool)

def prepare_semantic_frame(
    raw: pd.DataFrame, inferred: pd.DataFrame
) -> tuple[pd.DataFrame, pd.DataFrame]:
    frame = raw.copy()
    absence_masks = build_structural_absence_masks(raw)
    for column in frame.columns:
        raw_na = raw[column].eq("NA")
        if column in STRUCTURAL_ABSENCE:
            frame.loc[absence_masks[column], column] = "None"
            frame.loc[raw_na & ~absence_masks[column], column] = np.nan
        else:
            frame.loc[raw_na, column] = np.nan
    numeric_columns = inferred.select_dtypes(include=np.number).columns
    for column in numeric_columns:
        frame[column] = pd.to_numeric(frame[column], errors="coerce")
    return frame, absence_masks

train, structural_absence_train = prepare_semantic_frame(raw_train, inferred_train)
test, structural_absence_test = prepare_semantic_frame(raw_test, inferred_test)

assert train.shape == (1460, 81)
assert test.shape == (1459, 80)
assert set(train.columns) - set(test.columns) == {"SalePrice"}
assert train["Id"].is_unique and test["Id"].is_unique
assert train["SalePrice"].notna().all() and (train["SalePrice"] > 0).all()
assert int((raw_train["MasVnrType"] == "None").sum()) == 864
assert int((raw_train["MasVnrType"] == "NA").sum()) == 8

# Known data anomalies must remain unknown rather than becoming semantic absence.
assert pd.isna(train.loc[train["Id"].eq(949), "BsmtExposure"].item())
assert pd.isna(train.loc[train["Id"].eq(333), "BsmtFinType2"].item())
assert pd.isna(test.loc[test["Id"].eq(2127), "GarageFinish"].item())
assert test.loc[test["Id"].isin([2421, 2504, 2600]), "PoolQC"].isna().all()

print(f"train shape: {train.shape}; test shape: {test.shape}")
print(f"duplicate train rows: {train.duplicated().sum()}")
print(f"duplicate train Id: {train['Id'].duplicated().sum()}")
print("MasVnrType raw tokens:", raw_train["MasVnrType"].value_counts().head().to_dict())


train shape: (1460, 81); test shape: (1459, 80)
duplicate train rows: 0
duplicate train Id: 0
MasVnrType raw tokens: {'None': 864, 'BrkFace': 445, 'Stone': 128, 'BrkCmn': 15, 'NA': 8}


## 3. Feature audit

Storage dtype is not the same as modeling meaning. For example, `MSSubClass` and `MoSold` are numeric codes but should be handled as categorical features. The exported audit is the handoff contract for preprocessing.


In [3]:
ORDINAL_COLUMNS = {
    "OverallQual", "OverallCond", "ExterQual", "ExterCond", "BsmtQual",
    "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2", "HeatingQC",
    "KitchenQual", "Functional", "FireplaceQu", "GarageFinish", "GarageQual",
    "GarageCond", "PoolQC", "Fence", "LotShape", "LandSlope", "PavedDrive",
}
TEMPORAL_COLUMNS = {"YearBuilt", "YearRemodAdd", "GarageYrBlt", "YrSold"}
CATEGORICAL_CODES = {"MSSubClass", "MoSold"}
feature_columns = [c for c in train.columns if c not in {"Id", "SalePrice"}]

def semantic_type(column: str) -> str:
    if column in ORDINAL_COLUMNS:
        return "ordinal"
    if column in TEMPORAL_COLUMNS:
        return "temporal"
    if column in CATEGORICAL_CODES or not pd.api.types.is_numeric_dtype(train[column]):
        return "categorical"
    if train[column].nunique(dropna=True) <= 20:
        return "numeric_discrete"
    return "numeric_continuous"

def unknown_missing_mask(
    frame: pd.DataFrame, absence_masks: pd.DataFrame, column: str
) -> pd.Series:
    mask = frame[column].isna()
    if column in absence_masks:
        mask = mask & ~absence_masks[column]
    return mask

unknown_missing_train = pd.DataFrame({
    c: unknown_missing_mask(train, structural_absence_train, c) for c in feature_columns
})
unknown_missing_test = pd.DataFrame({
    c: unknown_missing_mask(test, structural_absence_test, c) for c in feature_columns
})

def missing_semantics(column: str) -> str:
    structural_count = (
        structural_absence_train.get(column, pd.Series(False, index=train.index)).sum()
        + structural_absence_test.get(column, pd.Series(False, index=test.index)).sum()
    )
    unknown_count = unknown_missing_train[column].sum() + unknown_missing_test[column].sum()
    if structural_count and unknown_count:
        return "mixed_structural_and_unknown"
    if structural_count:
        return "structural_absence"
    if unknown_count:
        return "unknown_missing"
    return "none"

feature_audit = pd.DataFrame({
    "feature": feature_columns,
    "storage_dtype": [str(train[c].dtype) for c in feature_columns],
    "semantic_type": [semantic_type(c) for c in feature_columns],
    "n_unique_train": [train[c].nunique(dropna=True) for c in feature_columns],
    "n_unique_test": [test[c].nunique(dropna=True) for c in feature_columns],
    "storage_nan_count_train": [train[c].isna().sum() for c in feature_columns],
    "storage_nan_count_test": [test[c].isna().sum() for c in feature_columns],
    "missing_count_train": [unknown_missing_train[c].sum() for c in feature_columns],
    "missing_pct_train": [unknown_missing_train[c].mean() * 100 for c in feature_columns],
    "missing_count_test": [unknown_missing_test[c].sum() for c in feature_columns],
    "missing_pct_test": [unknown_missing_test[c].mean() * 100 for c in feature_columns],
    "structural_absence_count_train": [
        structural_absence_train[c].sum() if c in structural_absence_train else 0
        for c in feature_columns
    ],
    "structural_absence_count_test": [
        structural_absence_test[c].sum() if c in structural_absence_test else 0
        for c in feature_columns
    ],
    "missing_semantics": [missing_semantics(c) for c in feature_columns],
    "raw_NA_count_train": [(raw_train[c] == "NA").sum() for c in feature_columns],
    "raw_NA_count_test": [(raw_test[c] == "NA").sum() for c in feature_columns],
    "zero_count_train": [
        (train[c] == 0).sum() if pd.api.types.is_numeric_dtype(train[c]) else 0
        for c in feature_columns
    ],
})
feature_audit.to_csv(METRICS_DIR / "feature_audit.csv", index=False)

assert len(feature_audit) == 79
print(feature_audit["semantic_type"].value_counts().to_string())
print()
print("Unknown missing values after row-level structural-absence mapping:")
print(feature_audit.loc[feature_audit["missing_count_train"] > 0,
                        ["feature", "missing_count_train", "missing_pct_train"]]
      .sort_values("missing_count_train", ascending=False).to_string(index=False))


semantic_type
categorical           26
ordinal               21
numeric_continuous    17
numeric_discrete      11
temporal               4

Unknown missing values after row-level structural-absence mapping:
     feature  missing_count_train  missing_pct_train
 LotFrontage                  259          17.739726
  MasVnrType                    8           0.547945
  MasVnrArea                    8           0.547945
BsmtExposure                    1           0.068493
BsmtFinType2                    1           0.068493
  Electrical                    1           0.068493


## 4. Target distribution

Kaggle evaluates error on the logarithm of prices, so both the original target and `log1p(SalePrice)` are inspected.


In [4]:
sale_price = train["SalePrice"]
log_sale_price = np.log1p(sale_price)
target_summary = pd.DataFrame({
    "original": sale_price.describe(),
    "log1p": log_sale_price.describe(),
})
target_shape = pd.DataFrame({
    "original": {"skewness": sale_price.skew(), "kurtosis": sale_price.kurt()},
    "log1p": {"skewness": log_sale_price.skew(), "kurtosis": log_sale_price.kurt()},
})
target_summary.to_csv(METRICS_DIR / "target_summary.csv")

print(target_summary.round(3).to_string())
print()
print("Shape statistics:")
print(target_shape.round(3).to_string())

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].hist(sale_price, bins=35, color="#1f6f8b", edgecolor="white")
axes[0].axvline(sale_price.median(), color="#c44e52", linestyle="--", label="Median")
axes[0].set(title="SalePrice distribution", xlabel="Sale price (USD)", ylabel="Homes")
axes[0].legend()
axes[1].hist(log_sale_price, bins=35, color="#d99445", edgecolor="white")
axes[1].axvline(log_sale_price.median(), color="#c44e52", linestyle="--", label="Median")
axes[1].set(title="log1p(SalePrice) distribution", xlabel="log1p(SalePrice)", ylabel="Homes")
axes[1].legend()
save_figure("eda_target_distribution.png")
plt.show()


         original     log1p
count    1460.000  1460.000
mean   180921.196    12.024
std     79442.503     0.399
min     34900.000    10.460
25%    129975.000    11.775
50%    163000.000    12.002
75%    214000.000    12.274
max    755000.000    13.534

Shape statistics:
          original  log1p
skewness     1.883  0.121
kurtosis     6.536  0.810
Saved: outputs\figures\eda_target_distribution.png
cell-8:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


## 5. Missing-value semantics

Raw `NA` rates describe tokens in the source file. Structural absence is assigned row by row from companion features; for example, basement categories become `None` only when `TotalBsmtSF == 0`. An `NA` attached to an existing basement, garage, fireplace, or pool remains unknown missing data.


In [5]:
raw_na_counts = pd.Series({c: int((raw_train[c] == "NA").sum()) for c in feature_columns})
semantic_missing_counts = unknown_missing_train.sum()
structural_counts = pd.Series({
    c: int(structural_absence_train[c].sum()) if c in structural_absence_train else 0
    for c in feature_columns
})
missing_audit = pd.DataFrame({
    "raw_NA_count_train": raw_na_counts,
    "raw_NA_pct_train": raw_na_counts / len(train) * 100,
    "structural_absence_count_train": structural_counts,
    "structural_absence_pct_train": structural_counts / len(train) * 100,
    "unknown_missing_count_train": semantic_missing_counts,
    "unknown_missing_pct_train": semantic_missing_counts / len(train) * 100,
    "unknown_missing_count_test": unknown_missing_test.sum(),
    "unknown_missing_pct_test": unknown_missing_test.mean() * 100,
}).sort_values("raw_NA_count_train", ascending=False)
missing_audit.to_csv(METRICS_DIR / "missing_audit.csv")

top_missing = missing_audit.loc[missing_audit["raw_NA_count_train"] > 0].head(20).sort_values("raw_NA_pct_train")
fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(top_missing.index, top_missing["raw_NA_pct_train"], color="#5b8e7d", alpha=0.75, label="Raw NA token")
ax.barh(top_missing.index, top_missing["unknown_missing_pct_train"], color="#c44e52", label="Unknown after semantic mapping")
ax.set(title="Raw NA tokens versus unknown missing values", xlabel="Percentage of training rows", ylabel="Feature")
ax.legend()
save_figure("eda_missing_rates.png")
plt.show()

print(f"Raw NA tokens: {int(raw_na_counts.sum())} across {(raw_na_counts > 0).sum()} columns")
print(f"Structural absences identified: {int(structural_counts.sum())} across {(structural_counts > 0).sum()} columns")
print(f"Unknown missing values: {int(semantic_missing_counts.sum())} across {(semantic_missing_counts > 0).sum()} columns")


Saved: outputs\figures\eda_missing_rates.png
cell-10:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
Raw NA tokens: 6965 across 19 columns
Structural absences identified: 6687 across 15 columns
Unknown missing values: 278 across 6 columns


In [6]:
group_columns = [
    "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2",
    "GarageType", "GarageYrBlt", "GarageFinish", "GarageQual", "GarageCond",
]
absence_matrix = pd.DataFrame({
    c: (raw_train[c] == "NA").astype(int) for c in group_columns
})
co_missing = absence_matrix.corr().fillna(0)

fig, ax = plt.subplots(figsize=(8, 7))
image = ax.imshow(co_missing, cmap="YlGnBu", vmin=0, vmax=1)
ax.set_xticks(range(len(group_columns)), group_columns, rotation=60, ha="right")
ax.set_yticks(range(len(group_columns)), group_columns)
ax.set_title("Co-occurrence of basement and garage absence")
fig.colorbar(image, ax=ax, label="Phi/Pearson correlation")
save_figure("eda_missing_cooccurrence.png")
plt.show()


Saved: outputs\figures\eda_missing_cooccurrence.png
cell-11:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


### Missing indicators versus target

The table below compares `log1p(SalePrice)` for homes flagged by structural absence or unknown missingness against all unflagged homes. It is diagnostic evidence for later indicator design, not an imputation decision.


In [7]:
missing_target_rows = []
for column in feature_columns:
    indicators = {}
    if column in structural_absence_train and structural_absence_train[column].any():
        indicators["structural_absence"] = structural_absence_train[column]
    if unknown_missing_train[column].any():
        indicators["unknown_missing"] = unknown_missing_train[column]
    for indicator_name, indicator in indicators.items():
        flagged = log_sale_price[indicator]
        unflagged = log_sale_price[~indicator]
        missing_target_rows.append({
            "feature": column,
            "indicator": indicator_name,
            "flagged_count": int(indicator.sum()),
            "unflagged_count": int((~indicator).sum()),
            "flagged_mean_log_target": flagged.mean(),
            "unflagged_mean_log_target": unflagged.mean(),
            "mean_log_target_delta": flagged.mean() - unflagged.mean(),
            "flagged_median_log_target": flagged.median(),
            "unflagged_median_log_target": unflagged.median(),
            "median_log_target_delta": flagged.median() - unflagged.median(),
        })

missing_target_effect = pd.DataFrame(missing_target_rows).sort_values(
    ["indicator", "median_log_target_delta"], key=lambda values: values.abs() if values.name == "median_log_target_delta" else values,
    ascending=[True, False],
)
missing_target_effect.to_csv(METRICS_DIR / "missing_target_effect.csv", index=False)

print("Largest absolute median log-target differences:")
print(missing_target_effect.reindex(
    missing_target_effect["median_log_target_delta"].abs().sort_values(ascending=False).index
).head(15).round(3).to_string(index=False))


Largest absolute median log-target differences:
     feature          indicator  flagged_count  unflagged_count  flagged_mean_log_target  unflagged_mean_log_target  mean_log_target_delta  flagged_median_log_target  unflagged_median_log_target  median_log_target_delta
BsmtFinType2    unknown_missing              1             1459                   12.557                     12.024                  0.533                     12.557                       12.002                    0.555
 GarageYrBlt structural_absence             81             1379                   11.492                     12.055                 -0.563                     11.513                       12.029                   -0.516
GarageFinish structural_absence             81             1379                   11.492                     12.055                 -0.563                     11.513                       12.029                   -0.516
  GarageQual structural_absence             81             1379         

## 6. Numeric feature relationships

Pearson captures linear relationships, while Spearman also captures monotonic relationships. Correlations are computed against `log1p(SalePrice)` to align with the competition metric.


In [8]:
numeric_features = [
    c for c in feature_columns
    if pd.api.types.is_numeric_dtype(train[c]) and semantic_type(c) not in {"categorical"}
]
pearson = train[numeric_features].corrwith(log_sale_price, method="pearson")
spearman = train[numeric_features].corrwith(log_sale_price, method="spearman")
target_correlations = pd.DataFrame({"pearson_log_target": pearson, "spearman_log_target": spearman})
target_correlations["max_abs"] = target_correlations.abs().max(axis=1)
target_correlations = target_correlations.sort_values("max_abs", ascending=False)
target_correlations.to_csv(METRICS_DIR / "target_correlations.csv")

top = target_correlations.head(15).sort_values("max_abs")
fig, ax = plt.subplots(figsize=(9, 7))
y = np.arange(len(top))
ax.barh(y - 0.18, top["pearson_log_target"], height=0.36, label="Pearson", color="#1f6f8b")
ax.barh(y + 0.18, top["spearman_log_target"], height=0.36, label="Spearman", color="#d99445")
ax.set_yticks(y, top.index)
ax.set(title="Top numeric relationships with log1p(SalePrice)", xlabel="Correlation", ylabel="Feature")
ax.legend()
save_figure("eda_target_correlations.png")
plt.show()

print(target_correlations.head(15).round(3).to_string())


Saved: outputs\figures\eda_target_correlations.png
cell-15:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
              pearson_log_target  spearman_log_target  max_abs
OverallQual                0.817                0.810    0.817
GrLivArea                  0.701                0.731    0.731
GarageCars                 0.681                0.691    0.691
YearBuilt                  0.587                0.653    0.653
GarageArea                 0.651                0.649    0.651
FullBath                   0.595                0.636    0.636
TotalBsmtSF                0.612                0.603    0.612
1stFlrSF                   0.597                0.575    0.597
GarageYrBlt                0.541                0.594    0.594
YearRemodAdd               0.566                0.571    0.571
TotRmsAbvGrd               0.534                0.533    0.534
Fireplaces                 0.489                0.519    0.519
OpenPorchSF                0.321            

In [9]:
scatter_features = ["OverallQual", "GrLivArea", "GarageCars", "TotalBsmtSF", "YearBuilt", "1stFlrSF"]
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, feature in zip(axes.flat, scatter_features):
    ax.scatter(train[feature], log_sale_price, s=13, alpha=0.45, color="#1f6f8b")
    ax.set(title=feature, xlabel=feature, ylabel="log1p(SalePrice)")
save_figure("eda_top_numeric_scatter.png")
plt.show()


Saved: outputs\figures\eda_top_numeric_scatter.png
cell-16:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


## 7. Categorical and ordinal relationships

Group medians are shown together with sample sizes so rare categories are not mistaken for stable effects.


In [10]:
neighborhood_stats = train.groupby("Neighborhood", observed=True)["SalePrice"].agg(["count", "median"])
neighborhood_stats = neighborhood_stats.sort_values("median")
quality_stats = train.groupby("OverallQual", observed=True)["SalePrice"].agg(["count", "median"])

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
axes[0].barh(neighborhood_stats.index, neighborhood_stats["median"], color="#5b8e7d")
axes[0].set(title="Median SalePrice by Neighborhood", xlabel="Median sale price (USD)", ylabel="Neighborhood")
axes[1].bar(quality_stats.index.astype(str), quality_stats["median"], color="#d99445")
axes[1].set(title="Median SalePrice by OverallQual", xlabel="Overall quality", ylabel="Median sale price (USD)")
save_figure("eda_categorical_target.png")
plt.show()

print("Neighborhoods with smallest sample sizes:")
print(neighborhood_stats.sort_values("count").head(8).to_string())


Saved: outputs\figures\eda_categorical_target.png
cell-18:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
Neighborhoods with smallest sample sizes:
              count    median
Neighborhood                 
Blueste           2  137500.0
NPkVill           9  146000.0
Veenker          11  218000.0
BrDale           16  106000.0
Blmngtn          17  191000.0
MeadowV          17   88000.0
StoneBr          25  278000.0
SWISU            25  139500.0


### Category-level coverage and rarity

Category levels are audited across train and test, including categorical numeric codes. Levels with fewer than five training rows are flagged as rare; train-only and test-only levels identify encoder edge cases.


In [11]:
RARE_CATEGORY_THRESHOLD = 5
category_columns = [
    c for c in feature_columns if semantic_type(c) in {"categorical", "ordinal"}
]

def normalized_levels(series: pd.Series, categorical_code: bool) -> pd.Series:
    values = series.astype("object").where(series.notna(), "<MISSING>")
    if categorical_code:
        return values.map(lambda value: str(int(value)) if isinstance(value, (int, float, np.integer, np.floating)) else str(value))
    return values.astype(str)

category_rows = []
for column in category_columns:
    train_levels = normalized_levels(train[column], column in CATEGORICAL_CODES)
    test_levels = normalized_levels(test[column], column in CATEGORICAL_CODES)
    train_counts = train_levels.value_counts()
    test_counts = test_levels.value_counts()
    for level in sorted(train_counts.index.union(test_counts.index)):
        train_count = int(train_counts.get(level, 0))
        test_count = int(test_counts.get(level, 0))
        status = "shared" if train_count and test_count else "train_only" if train_count else "test_only"
        category_rows.append({
            "feature": column,
            "level": level,
            "train_count": train_count,
            "test_count": test_count,
            "train_pct": train_count / len(train) * 100,
            "test_pct": test_count / len(test) * 100,
            "status": status,
            "rare_train": 0 < train_count < RARE_CATEGORY_THRESHOLD,
        })

category_level_audit = pd.DataFrame(category_rows).sort_values(["feature", "status", "train_count", "level"])
category_level_audit.to_csv(METRICS_DIR / "category_level_audit.csv", index=False)

print("Category-level status counts:")
print(category_level_audit["status"].value_counts().to_string())
print(f"Rare train levels (<{RARE_CATEGORY_THRESHOLD} rows): {category_level_audit['rare_train'].sum()}")
print("Test-only levels:")
print(category_level_audit.loc[category_level_audit["status"].eq("test_only"),
                               ["feature", "level", "test_count"]].to_string(index=False))


Category-level status counts:
status
shared        297
train_only     19
test_only      15
Rare train levels (<5 rows): 54
Test-only levels:
     feature     level  test_count
    BsmtCond <MISSING>           4
BsmtFinType1 <MISSING>           1
    BsmtQual <MISSING>           3
 Exterior1st <MISSING>           1
 Exterior2nd <MISSING>           1
  Functional <MISSING>           2
  GarageCond <MISSING>           2
GarageFinish <MISSING>           2
  GarageQual <MISSING>           2
 KitchenQual <MISSING>           1
  MSSubClass       150           1
    MSZoning <MISSING>           4
      PoolQC <MISSING>           3
    SaleType <MISSING>           1
   Utilities <MISSING>           2


## 8. Redundancy among numeric features

High pairwise correlation does not automatically justify removing a feature, but it identifies redundant measurements for baseline diagnostics and regularization decisions.


In [12]:
heatmap_features = target_correlations.head(12).index.tolist()
feature_corr = train[heatmap_features].corr()
fig, ax = plt.subplots(figsize=(10, 8))
image = ax.imshow(feature_corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(heatmap_features)), heatmap_features, rotation=60, ha="right")
ax.set_yticks(range(len(heatmap_features)), heatmap_features)
ax.set_title("Correlation among leading numeric features")
fig.colorbar(image, ax=ax, label="Pearson correlation")
save_figure("eda_numeric_redundancy.png")
plt.show()

pairs = feature_corr.where(np.triu(np.ones(feature_corr.shape), k=1).astype(bool)).stack()
print("Strongest absolute feature-feature correlations:")
print(pairs.reindex(pairs.abs().sort_values(ascending=False).index).head(10).round(3).to_string())


Saved: outputs\figures\eda_numeric_redundancy.png
cell-22:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
Strongest absolute feature-feature correlations:
GarageCars   GarageArea      0.882
YearBuilt    GarageYrBlt     0.826
GrLivArea    TotRmsAbvGrd    0.825
TotalBsmtSF  1stFlrSF        0.820
GarageYrBlt  YearRemodAdd    0.642
GrLivArea    FullBath        0.630
OverallQual  GarageCars      0.601
             GrLivArea       0.593
YearBuilt    YearRemodAdd    0.593
GarageCars   GarageYrBlt     0.589


## 9. Outlier review

The paper highlights very large homes. These observations are labeled for review but are not removed in EDA.


In [13]:
large_homes = train.loc[train["GrLivArea"] > 4000, ["Id", "GrLivArea", "SalePrice"]].copy()
fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(train["GrLivArea"], train["SalePrice"], s=18, alpha=0.5, color="#1f6f8b")
ax.scatter(large_homes["GrLivArea"], large_homes["SalePrice"], s=65, color="#c44e52", label="GrLivArea > 4,000")
for row in large_homes.itertuples():
    ax.annotate(f"Id {row.Id}", (row.GrLivArea, row.SalePrice), xytext=(5, 5), textcoords="offset points")
ax.set(title="GrLivArea versus SalePrice", xlabel="Above-ground living area (sq ft)", ylabel="Sale price (USD)")
ax.legend()
save_figure("eda_grlivarea_outliers.png")
plt.show()

print(large_homes.to_string(index=False))


Saved: outputs\figures\eda_grlivarea_outliers.png
    Id  GrLivArea  SalePrice
 524.0     4676.0   184750.0
 692.0     4316.0   755000.0
1183.0     4476.0   745000.0
1299.0     5642.0   160000.0


## 10. Train/test distribution comparison

Standardized mean difference (SMD) summarizes numeric shifts. Total variation distance (TVD) summarizes categorical frequency shifts. These are diagnostics only; no preprocessing parameter is fitted on test data.


In [14]:
common_numeric = [c for c in numeric_features if c in test.columns]
smd = {}
for column in common_numeric:
    scale = train[column].std()
    smd[column] = 0.0 if not np.isfinite(scale) or scale == 0 else abs(train[column].mean() - test[column].mean()) / scale
smd = pd.Series(smd, name="shift_score").sort_values(ascending=False)

common_categorical = [c for c in feature_columns if semantic_type(c) in {"categorical", "ordinal"} and c in test.columns]
tvd = {}
for column in common_categorical:
    train_freq = train[column].fillna("<MISSING>").astype(str).value_counts(normalize=True)
    test_freq = test[column].fillna("<MISSING>").astype(str).value_counts(normalize=True)
    levels = train_freq.index.union(test_freq.index)
    tvd[column] = 0.5 * (train_freq.reindex(levels, fill_value=0) - test_freq.reindex(levels, fill_value=0)).abs().sum()
tvd = pd.Series(tvd, name="shift_score").sort_values(ascending=False)

drift_audit = pd.concat([
    pd.DataFrame({"feature": smd.index, "feature_type": "numeric", "shift_score": smd.values}),
    pd.DataFrame({"feature": tvd.index, "feature_type": "categorical_or_ordinal", "shift_score": tvd.values}),
], ignore_index=True)
drift_audit.to_csv(METRICS_DIR / "train_test_drift.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
smd.head(12).sort_values().plot.barh(ax=axes[0], color="#1f6f8b")
axes[0].set(title="Largest numeric train/test shifts", xlabel="Absolute SMD", ylabel="Feature")
tvd.head(12).sort_values().plot.barh(ax=axes[1], color="#d99445")
axes[1].set(title="Largest categorical train/test shifts", xlabel="Total variation distance", ylabel="Feature")
save_figure("eda_train_test_shift.png")
plt.show()

print("Top numeric shifts:")
print(smd.head(8).round(3).to_string())
print()
print("Top categorical/ordinal shifts:")
print(tvd.head(8).round(3).to_string())


Saved: outputs\figures\eda_train_test_shift.png
cell-26:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
Top numeric shifts:
TotRmsAbvGrd    0.082
LotArea         0.070
LotFrontage     0.061
YearRemodAdd    0.058
GrLivArea       0.056
3SsnPorch       0.055
Fireplaces      0.049
2ndFlrSF        0.048

Top categorical/ordinal shifts:
Neighborhood    0.072
MoSold          0.060
OverallQual     0.045
FireplaceQu     0.037
Exterior2nd     0.032
BsmtFinType1    0.032
MSSubClass      0.030
MSZoning        0.029


## 11. Reproducibility checks and handoff summary

The assertions below verify artifact coverage. Interpretive conclusions are maintained in `memory-bank/eda-findings.md` so preprocessing decisions remain traceable.


In [15]:
expected_figures = {
    "eda_target_distribution.png", "eda_missing_rates.png", "eda_missing_cooccurrence.png",
    "eda_target_correlations.png", "eda_top_numeric_scatter.png", "eda_categorical_target.png",
    "eda_numeric_redundancy.png", "eda_grlivarea_outliers.png", "eda_train_test_shift.png",
}
expected_metrics = {
    "feature_audit.csv", "target_summary.csv", "missing_audit.csv", "missing_target_effect.csv",
    "target_correlations.csv", "category_level_audit.csv", "train_test_drift.csv",
}

missing_figures = expected_figures - {p.name for p in FIGURES_DIR.glob("eda_*.png")}
missing_metrics = expected_metrics - {p.name for p in METRICS_DIR.glob("*.csv")}
assert not missing_figures, f"Missing figures: {sorted(missing_figures)}"
assert not missing_metrics, f"Missing metrics: {sorted(missing_metrics)}"
assert len(pd.read_csv(METRICS_DIR / "feature_audit.csv")) == 79

category_check = pd.read_csv(METRICS_DIR / "category_level_audit.csv", dtype={"level": str})
assert ((category_check["feature"] == "MSSubClass")
        & (category_check["level"] == "150")
        & (category_check["status"] == "test_only")).any()
assert set(raw_train.loc[raw_train["BsmtExposure"].eq("NA"), "Id"]) == set(
    train.loc[train["BsmtExposure"].eq("None") | train["BsmtExposure"].isna(), "Id"]
)
assert feature_audit.set_index("feature").loc["BsmtExposure", "missing_count_train"] == 1
assert feature_audit.set_index("feature").loc["BsmtFinType2", "missing_count_train"] == 1

summary = {
    "train_rows": len(train),
    "test_rows": len(test),
    "features_audited": len(feature_audit),
    "unknown_missing_columns_train": int((semantic_missing_counts > 0).sum()),
    "category_levels_audited": len(category_level_audit),
    "missing_target_indicators": len(missing_target_effect),
    "saleprice_skew": round(float(sale_price.skew()), 3),
    "log1p_saleprice_skew": round(float(log_sale_price.skew()), 3),
    "large_home_candidates": large_homes["Id"].tolist(),
    "figures_written": len(expected_figures),
    "metric_tables_written": len(expected_metrics),
}
print(pd.Series(summary).to_string())


train_rows                                                 1460
test_rows                                                  1459
features_audited                                             79
unknown_missing_columns_train                                 6
category_levels_audited                                     331
missing_target_indicators                                    21
saleprice_skew                                            1.883
log1p_saleprice_skew                                      0.121
large_home_candidates            [524.0, 692.0, 1183.0, 1299.0]
figures_written                                               9
metric_tables_written                                         7
